# AI-Based Adaptive Mock Interview System — v10

**Dataset**: `interview_dataset_v10_FINAL.json` (1,026 rows · 9 skills · 3 difficulty levels · JSONL format)

**Skills covered**: Python, OOP, DBMS, DSA, ML, CN, OS, System Design, HR

**Key updates in this version**:
- Dataset upgraded from 1,663-row CSV → 1,026-row JSONL (`interview_dataset_v10_FINAL.json`)
- JSONL loader replaces `pd.read_csv` — handles one JSON object per line
- Pre-computed features upgraded: uses `bert_score` + `tfidf_score` (alongside `cos_similarity`, `length_ratio`, `aligned_score`, `word_count`)
- `bert_score` + `tfidf_score` added as numeric features — improves ML pipeline signal
- `ColumnTransformer` retains TF-IDF on raw `answer` text + all 6 numeric features
- `ComplementNB` uses `MinMaxScaler` (non-negative requirement); LR / SVM / RF use `StandardScaler`
- Best model auto-selected by weighted F1 across Stratified 5-Fold CV
- NLP Scorer uses `sentence-transformers` for semantic similarity (falls back gracefully)
- Question bank built from 508 unique questions with ideal answers & keyword extraction
- Adaptive difficulty (Easy → Medium → Hard) driven by final score

In [ ]:
# =========================
# MODULE 1: Install & Import
# =========================
!pip install -q scikit-learn pandas matplotlib sentence-transformers

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import re
import json
import joblib

from sklearn.model_selection import cross_val_score, cross_validate, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB

print("All imports successful.")

In [ ]:
# =========================
# MODULE 2: Load Dataset
# =========================
# Dataset format: JSONL — one JSON object per line
# Upload your file if running in Colab:
# from google.colab import files
# files.upload()

DATASET_PATH = "interview_dataset_v10_FINAL.json"

records = []
try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    df = pd.DataFrame(records)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    raise SystemExit(f"ERROR: '{DATASET_PATH}' not found. Upload the file and retry.")

# ── Sanity checks ──────────────────────────────────────────────
print("\nColumns      :", df.columns.tolist())
print("Shape        :", df.shape)
print("\nLabel Distribution:")
print(df['label'].value_counts())
print("\nBalance ratio:", round(df['label'].value_counts().max() /
                                  df['label'].value_counts().min(), 2), "(target < 1.5)")
print("\nSkills       :", df['skill'].unique().tolist())
print("Difficulties :", df['difficulty'].unique().tolist())
print("\nUnique questions:", df['question'].nunique())
print("\nPer skill-difficulty question count:")
print(df.groupby(['skill', 'difficulty'])['question'].nunique().to_string())

# ── Feature completeness check ─────────────────────────────────
required_cols = ['answer', 'ideal_answer', 'label', 'skill', 'difficulty',
                 'bert_score', 'tfidf_score', 'cos_similarity',
                 'length_ratio', 'aligned_score', 'word_count']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"\nWARNING — missing columns: {missing}")
else:
    print("\nAll required columns present. ✓")

In [ ]:
# =========================
# MODULE 3: NLP Scoring Engine
# =========================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
    USE_EMBEDDINGS = True
    print("Semantic embeddings: ENABLED  (multi-qa-MiniLM-L6-cos-v1)")
except ImportError:
    USE_EMBEDDINGS = False
    print("Semantic embeddings: DISABLED — install sentence-transformers to enable")


class NLPScorer:
    """
    Scores a candidate's answer against the ideal answer using four signals:
      1. Keyword match   (20 %)
      2. Length heuristic (10 %)
      3. TF-IDF cosine similarity (30 % | 70 % without embeddings)
      4. Semantic similarity via sentence-transformers (40 % when available)
    """

    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words='english')

    def score_keyword_match(self, answer: str, expected_keywords: list) -> float:
        if not expected_keywords:
            return 1.0
        text = answer.lower()
        matched = sum(1 for kw in expected_keywords
                      if re.search(r'\b' + re.escape(kw.lower()) + r'\b', text))
        return matched / len(expected_keywords)

    def score_length(self, answer: str, ideal_length: int = 20) -> float:
        return min(len(answer.split()) / ideal_length, 1.0)

    def score_tfidf_similarity(self, answer: str, golden_answer: str) -> float:
        try:
            tfidf = self.vectorizer.fit_transform([golden_answer, answer])
            return float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0])
        except ValueError:
            return 0.0

    def score_semantic_similarity(self, answer: str, golden_answer: str) -> float:
        if not USE_EMBEDDINGS:
            return 0.0
        e1 = np.array(embedder.encode(golden_answer)).reshape(1, -1)
        e2 = np.array(embedder.encode(answer)).reshape(1, -1)
        return float(max(0.0, cosine_similarity(e1, e2)[0][0]))

    def compute_overall_score(self, answer: str, golden_answer: str,
                               expected_keywords: list = None) -> float:
        """Returns a score from 0–100."""
        if not answer.strip():
            return 0.0
        if expected_keywords is None:
            expected_keywords = []

        kw     = self.score_keyword_match(answer, expected_keywords)
        length = self.score_length(answer, ideal_length=20)
        tfidf  = self.score_tfidf_similarity(answer, golden_answer)

        if USE_EMBEDDINGS:
            semantic = self.score_semantic_similarity(answer, golden_answer)
            total = kw * 0.20 + length * 0.10 + tfidf * 0.30 + semantic * 0.40
        else:
            semantic = 0.0
            total = kw * 0.20 + length * 0.10 + tfidf * 0.70

        return round(total * 100, 2)

    def get_feature_scores(self, answer: str, golden_answer: str,
                            expected_keywords: list = None):
        """Returns (cos_sim, length_ratio, aligned_score, word_count) for ML prediction."""
        if expected_keywords is None:
            expected_keywords = []
        if USE_EMBEDDINGS:
            cos_sim = self.score_semantic_similarity(answer, golden_answer)
        else:
            cos_sim = self.score_tfidf_similarity(answer, golden_answer)
        golden_words  = max(1, len(golden_answer.split()))
        length_ratio  = len(answer.split()) / golden_words
        aligned_score = self.score_keyword_match(answer, expected_keywords)
        tfidf_score   = self.score_tfidf_similarity(answer, golden_answer)
        word_count    = len(answer.split())
        return cos_sim, length_ratio, aligned_score, tfidf_score, word_count


print("NLPScorer defined successfully.")

In [ ]:
# =========================
# MODULE 4: Model Training & Comparison
# =========================
# Features used for classification:
#   Text   : raw 'answer' → TF-IDF (ngram 1-2, sublinear_tf)
#   Numeric: bert_score, tfidf_score, cos_similarity, length_ratio, aligned_score, word_count
#
# 'bert_score'  = pre-computed semantic similarity from dataset (sentence-transformers)
# 'tfidf_score' = pre-computed TF-IDF cosine similarity from dataset
# These pre-computed scores give the model strong signal without runtime embedding calls.

NUMERIC_FEATURES = ['bert_score', 'tfidf_score', 'cos_similarity',
                    'length_ratio', 'aligned_score', 'word_count']
TEXT_FEATURE     = 'answer'

X = df[['answer'] + NUMERIC_FEATURES].copy()
y = df['label']   # 'Good' | 'Average' | 'Poor'

# ── Preprocessors ──────────────────────────────────────────────
preprocessor_standard = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                               stop_words='english', max_features=5000), TEXT_FEATURE),
    ("num",   StandardScaler(),                                           NUMERIC_FEATURES)
])

preprocessor_minmax = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                               stop_words='english', max_features=5000), TEXT_FEATURE),
    ("num",   MinMaxScaler(),                                             NUMERIC_FEATURES)
])

# ── Model arena ────────────────────────────────────────────────
pipelines = {
    "Logistic Regression": Pipeline([
        ("pre", preprocessor_standard),
        ("clf", LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    "SVM": Pipeline([
        ("pre", preprocessor_standard),
        ("clf", SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocessor_standard),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                        max_depth=None, random_state=42))
    ]),
    "Complement NB": Pipeline([
        ("pre", preprocessor_minmax),
        ("clf", ComplementNB())
    ])
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']

print("\n===== CROSS VALIDATION (Stratified 5-Fold) =====")
final_results = {}

for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    final_results[name] = {
        "Accuracy" : round(scores['test_accuracy'].mean(),        4),
        "Precision": round(scores['test_precision_weighted'].mean(), 4),
        "Recall"   : round(scores['test_recall_weighted'].mean(), 4),
        "F1 Score" : round(scores['test_f1_weighted'].mean(),     4),
    }
    print(f"  {name:22s}  Acc: {final_results[name]['Accuracy']:.4f} "
          f" F1: {final_results[name]['F1 Score']:.4f}")

comparison_df = pd.DataFrame(final_results).T
print("\n===== FINAL COMPARISON TABLE =====")
print(comparison_df)

best_model_name = comparison_df['F1 Score'].idxmax()
print(f"\n*** Best model: {best_model_name} "
      f"(F1 = {comparison_df.loc[best_model_name, 'F1 Score']:.4f}) ***")

In [ ]:
# =========================
# MODULE 5: Confusion Matrices
# =========================
fig, axes = plt.subplots(1, len(pipelines), figsize=(5 * len(pipelines), 4))

LABELS = ['Good', 'Average', 'Poor']

for ax, (name, pipe) in zip(axes, pipelines.items()):
    y_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=-1)
    cm = confusion_matrix(y, y_pred, labels=LABELS)
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)

plt.suptitle("Confusion Matrices — Stratified 5-Fold CV (v10 Dataset)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices_v10.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: confusion_matrices_v10.png")

In [ ]:
# =========================
# MODULE 6: Bar Chart (HTML — renders in Colab / Jupyter)
# =========================
import json as _json
from IPython.display import HTML

chart_data = {
    "models" : comparison_df.index.tolist(),
    "metrics": {
        "Accuracy" : comparison_df["Accuracy"].tolist(),
        "Precision": comparison_df["Precision"].tolist(),
        "Recall"   : comparison_df["Recall"].tolist(),
        "F1 Score" : comparison_df["F1 Score"].tolist()
    },
    "best": best_model_name
}

html = f"""
<!DOCTYPE html><html><head>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
</head><body style="font-family:sans-serif;padding:20px;">

<div style="display:flex;flex-wrap:wrap;gap:12px;margin-bottom:1.5rem;">
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:140px;">
    <div style="font-size:12px;color:#888;margin-bottom:4px;">Best Model</div>
    <div style="font-size:17px;font-weight:600;" id="best-model">—</div>
  </div>
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:120px;">
    <div style="font-size:12px;color:#888;margin-bottom:4px;">Top Accuracy</div>
    <div style="font-size:17px;font-weight:600;" id="top-acc">—</div>
  </div>
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:120px;">
    <div style="font-size:12px;color:#888;margin-bottom:4px;">Top F1</div>
    <div style="font-size:17px;font-weight:600;" id="top-f1">—</div>
  </div>
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:120px;">
    <div style="font-size:12px;color:#888;margin-bottom:4px;">Dataset</div>
    <div style="font-size:17px;font-weight:600;">1,026 rows</div>
  </div>
</div>

<div style="display:flex;flex-wrap:wrap;gap:16px;margin-bottom:12px;font-size:12px;color:#666;">
  <span><span style="display:inline-block;width:10px;height:10px;border-radius:2px;background:#3266ad;margin-right:4px;"></span>Accuracy</span>
  <span><span style="display:inline-block;width:10px;height:10px;border-radius:2px;background:#73726c;margin-right:4px;"></span>Precision</span>
  <span><span style="display:inline-block;width:10px;height:10px;border-radius:2px;background:#1d9e75;margin-right:4px;"></span>Recall</span>
  <span><span style="display:inline-block;width:10px;height:10px;border-radius:2px;background:#d85a30;margin-right:4px;"></span>F1 Score</span>
</div>

<div style="position:relative;width:100%;height:360px;">
  <canvas id="modelChart"></canvas>
</div>

<div style="margin-top:1.5rem;overflow-x:auto;">
  <table style="width:100%;border-collapse:collapse;font-size:13px;" id="metrics-table"></table>
</div>

<script>
const data = {_json.dumps(chart_data)};
const colors = ['#3266ad','#73726c','#1d9e75','#d85a30'];
const metricKeys = Object.keys(data.metrics);
const datasets = metricKeys.map((k,i) => ({{
  label: k, data: data.metrics[k], backgroundColor: colors[i],
  borderRadius: 4, borderSkipped: false, barPercentage: 0.82, categoryPercentage: 0.78
}}));
new Chart(document.getElementById('modelChart'), {{
  type: 'bar',
  data: {{ labels: data.models, datasets }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ font: {{ size: 12 }}, color: '#888' }} }},
      y: {{ min: 0.5, max: 1.0, grid: {{ color: 'rgba(128,128,128,0.15)' }},
           ticks: {{ font: {{ size: 11 }}, color: '#888', callback: v => v.toFixed(2) }} }}
    }}
  }}
}});
const accVals = data.metrics['Accuracy'];
const f1Vals  = data.metrics['F1 Score'];
document.getElementById('best-model').textContent = data.best;
document.getElementById('top-acc').textContent    = Math.max(...accVals).toFixed(3);
document.getElementById('top-f1').textContent     = Math.max(...f1Vals).toFixed(3);
const table   = document.getElementById('metrics-table');
const hdrCols = ['Model', ...metricKeys];
let thtml = '<thead><tr>' + hdrCols.map(h =>
  `<th style="text-align:${{h==='Model'?'left':'right'}};padding:8px 12px;border-bottom:1px solid #ddd;color:#888;font-weight:500;">${{h}}</th>`
).join('') + '</tr></thead><tbody>';
data.models.forEach((m, mi) => {{
  const isTop = m === data.best;
  thtml += `<tr style="background:${{isTop?'#f9f9f9':'transparent'}}">`;
  thtml += `<td style="padding:8px 12px;border-bottom:0.5px solid #eee;font-weight:${{isTop?600:400}}">${{m}}${{isTop?' ★':''}}</td>`;
  metricKeys.forEach(k => {{
    thtml += `<td style="text-align:right;padding:8px 12px;border-bottom:0.5px solid #eee;">${{data.metrics[k][mi].toFixed(3)}}</td>`;
  }});
  thtml += '</tr>';
}});
table.innerHTML = thtml + '</tbody>';
</script></body></html>
"""

display(HTML(html))

In [ ]:
# =========================
# MODULE 7: Export Best Model
# =========================
best_pipe = pipelines[best_model_name]
best_pipe.fit(X, y)
joblib.dump(best_pipe, 'best_interview_model.pkl')
print(f"Best model '{best_model_name}' trained on full dataset.")
print("Saved: best_interview_model.pkl")

In [ ]:
# =========================
# MODULE 8: Build Question Bank from Dataset
# =========================
# Unique questions only — each entry carries:
#   question      : the interview question text
#   golden_answer : ideal_answer from dataset (ground truth)
#   keywords      : top content words from ideal_answer for keyword-match scoring

question_bank = {}   # { skill: { difficulty: [ {question, golden_answer, keywords}, ... ] } }

q_df = df[['skill', 'difficulty', 'question', 'ideal_answer']].drop_duplicates('question')


def extract_keywords(text: str, top_n: int = 8) -> list:
    """Top-N content words from ideal answer by raw frequency (fast, no TF-IDF corpus needed)."""
    STOPWORDS = {
        'this', 'that', 'with', 'from', 'have', 'been', 'they', 'them', 'when',
        'will', 'also', 'both', 'each', 'more', 'most', 'into', 'than', 'then',
        'over', 'used', 'uses', 'using', 'which', 'where', 'while', 'their',
        'there', 'these', 'those', 'your', 'like', 'such', 'only', 'some',
        'data', 'make', 'made', 'allows', 'means', 'type', 'types', 'example'
    }
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    freq  = {}
    for w in words:
        if w not in STOPWORDS:
            freq[w] = freq.get(w, 0) + 1
    return [w for w, _ in sorted(freq.items(), key=lambda x: -x[1])][:top_n]


for _, row in q_df.iterrows():
    skill = row['skill']
    diff  = row['difficulty']
    question_bank.setdefault(skill, {}).setdefault(diff, []).append({
        'question'     : row['question'],
        'golden_answer': row['ideal_answer'],
        'keywords'     : extract_keywords(row['ideal_answer'])
    })

SKILLS = sorted(question_bank.keys())
print(f"Question bank built — {len(q_df)} unique questions across {len(SKILLS)} skills.")
print(f"\nSkills: {SKILLS}")
print("\nQuestion counts per skill/difficulty:")
for skill in SKILLS:
    counts = {d: len(qs) for d, qs in question_bank[skill].items()}
    print(f"  {skill:15s}: {counts}")

In [ ]:
# =========================
# MODULE 9: Adaptive Interview Engine
# =========================

def get_next_question(skill, difficulty, asked_questions, question_bank):
    """Return a fresh question for the given skill/difficulty.
    Falls back to other difficulty levels within the same skill if pool is exhausted."""
    pool  = question_bank.get(skill, {}).get(difficulty, [])
    fresh = [q for q in pool if q['question'] not in asked_questions]
    if fresh:
        return random.choice(fresh)
    # Fallback: any unseen question for this skill
    all_skill_qs = [q for d, qs in question_bank.get(skill, {}).items()
                    for q in qs if q['question'] not in asked_questions]
    if all_skill_qs:
        return random.choice(all_skill_qs)
    # Last resort: allow repeat
    return random.choice(pool) if pool else None


def adapt_difficulty(current_difficulty: str, score: float) -> str:
    """C2 Elo-style adaptive engine — adjusts difficulty based on final score."""
    levels = ["Easy", "Medium", "Hard"]
    idx    = levels.index(current_difficulty)
    if score >= 70 and idx < 2:
        print("  [C2 Adaptation] Great answer! ↑ Increasing difficulty.")
        return levels[idx + 1]
    elif score <= 40 and idx > 0:
        print("  [C2 Adaptation] Needs improvement. ↓ Decreasing difficulty.")
        return levels[idx - 1]
    else:
        print("  [C2 Adaptation] Solid answer. → Maintaining difficulty.")
        return current_difficulty


def predict_label(model, answer: str, cos_sim: float, length_ratio: float,
                   aligned_score: float, tfidf_score: float, word_count: int) -> str:
    """Run trained ML model to classify the answer quality."""
    try:
        # bert_score at inference: use cos_sim (live semantic score) as proxy
        row = pd.DataFrame([{
            'answer'       : answer,
            'bert_score'   : cos_sim,         # live semantic similarity → bert_score proxy
            'tfidf_score'  : tfidf_score,
            'cos_similarity': cos_sim,
            'length_ratio' : length_ratio,
            'aligned_score': aligned_score,
            'word_count'   : word_count
        }])
        return model.predict(row)[0]
    except Exception as e:
        print(f"  ML Prediction Error: {e}")
        return "Unknown"


def label_to_score_adjustment(ml_label: str) -> float:
    return {"Good": +10, "Average": 0, "Poor": -10}.get(ml_label, 0)


def generate_feedback(nlp_score: float, ml_label: str, length_ratio: float,
                       aligned_score: float) -> str:
    """C4 Rule-based feedback — zero hallucination, pure if/else logic."""
    if ml_label == "Good" and nlp_score >= 70:
        return "Excellent! Your answer is comprehensive and accurate."
    elif ml_label == "Good" and nlp_score < 70:
        return "Good coverage — try to add a brief concrete example to strengthen your answer."
    elif ml_label == "Average" and length_ratio < 0.4:
        return "You are on the right track conceptually, but your answer is too brief. Expand further."
    elif ml_label == "Average" and aligned_score < 0.3:
        return "Partially correct — you are missing key technical terms. Review the ideal answer."
    elif ml_label == "Average":
        return "Decent answer. Include more depth and specific examples to reach 'Good'."
    elif ml_label == "Poor" and length_ratio < 0.2:
        return "Answer is too short to evaluate properly. Please give a more detailed response."
    elif ml_label == "Poor":
        return "This needs significant improvement. Study the ideal answer below carefully."
    return "Answer recorded."


def run_interview(question_bank, ml_model, scorer, total_rounds: int = 5):
    print("\n" + "="*58)
    print("   AI-Based Adaptive Mock Interview System  [v10]")
    print("="*58)

    skills = sorted(question_bank.keys())
    print("\nAvailable skills:")
    for i, s in enumerate(skills, 1):
        print(f"  {i:2d}. {s}")

    while True:
        try:
            choice = int(input("\nSelect skill number: "))
            if 1 <= choice <= len(skills):
                selected_skill = skills[choice - 1]
                break
            print(f"  Enter a number between 1 and {len(skills)}.")
        except ValueError:
            print("  Invalid input — enter a number.")

    print(f"\nSelected skill : {selected_skill}")
    print(f"Starting level : Medium  |  Rounds: {total_rounds}")
    print("-"*58)

    current_difficulty = "Medium"
    asked_questions    = set()
    session_log        = []

    for i in range(1, total_rounds + 1):
        print(f"\nRound {i}/{total_rounds}  |  Skill: {selected_skill}  |  Level: [{current_difficulty}]")

        q_data = get_next_question(selected_skill, current_difficulty, asked_questions, question_bank)
        if q_data is None:
            print("No more questions available. Ending interview.")
            break

        asked_questions.add(q_data['question'])
        print(f"\nQ: {q_data['question']}")
        print("Your answer (or 'skip' / 'quit'):")

        try:
            user_ans = input("> ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nExiting...")
            break

        if user_ans.lower() in ['quit', 'exit', 'q']:
            print("Interview ended early.")
            break

        if user_ans.lower() == 'skip':
            print("  Skipped.")
            session_log.append({'round': i, 'difficulty': current_difficulty,
                                'question': q_data['question'], 'nlp_score': 0,
                                'ml_label': 'Skipped', 'final_score': 0, 'feedback': 'Skipped'})
            continue

        print("\n  Evaluating...")

        # ── NLP scoring ─────────────────────────────────────────
        nlp_score = scorer.compute_overall_score(
            answer            = user_ans,
            golden_answer     = q_data['golden_answer'],
            expected_keywords = q_data['keywords']
        )

        # ── Feature extraction for ML ────────────────────────────
        cos_sim, length_ratio, aligned, tfidf_sc, word_count = scorer.get_feature_scores(
            user_ans, q_data['golden_answer'], q_data['keywords']
        )

        # ── ML classification ────────────────────────────────────
        ml_label = predict_label(ml_model, user_ans, cos_sim,
                                  length_ratio, aligned, tfidf_sc, word_count)

        # ── Combined final score ─────────────────────────────────
        final_score = round(min(100, max(0, nlp_score + label_to_score_adjustment(ml_label))), 2)

        # ── C4 Rule-based feedback ───────────────────────────────
        feedback = generate_feedback(nlp_score, ml_label, length_ratio, aligned)

        # ── Display results ──────────────────────────────────────
        print(f"\n  {'='*40}")
        print(f"  NLP Score     : {nlp_score:.1f} / 100")
        print(f"  ML Label      : {ml_label}")
        print(f"  Final Score   : {final_score:.1f} / 100")
        print(f"  Feedback      : {feedback}")
        print(f"  Ideal Answer  : {q_data['golden_answer'][:200]}...")
        print(f"  {'='*40}")

        session_log.append({
            'round'      : i,
            'difficulty' : current_difficulty,
            'question'   : q_data['question'],
            'nlp_score'  : nlp_score,
            'ml_label'   : ml_label,
            'final_score': final_score,
            'feedback'   : feedback
        })

        if i < total_rounds:
            current_difficulty = adapt_difficulty(current_difficulty, final_score)

    # ── Session summary ──────────────────────────────────────────
    print("\n" + "="*58)
    print("   SESSION SUMMARY")
    print("="*58)
    summary_df = pd.DataFrame(session_log)
    if not summary_df.empty:
        print(summary_df[['round', 'difficulty', 'ml_label', 'final_score']].to_string(index=False))
        answered = summary_df[summary_df['ml_label'] != 'Skipped']
        if not answered.empty:
            avg = answered['final_score'].mean()
            print(f"\n  Average Score    : {avg:.1f} / 100")
            print(f"  Good answers     : {(answered['ml_label']=='Good').sum()}")
            print(f"  Average answers  : {(answered['ml_label']=='Average').sum()}")
            print(f"  Poor answers     : {(answered['ml_label']=='Poor').sum()}")
    print("\nThank you for participating!")
    return summary_df


print("Interview engine ready — run the next cell to start.")

In [ ]:
# =========================
# MODULE 10: Start Interview
# =========================
scorer   = NLPScorer()
ml_model = joblib.load('best_interview_model.pkl')

session_results = run_interview(
    question_bank = question_bank,
    ml_model      = ml_model,
    scorer        = scorer,
    total_rounds  = 5    # change to run more / fewer rounds
)